In [79]:
import pandas as pd

attack = pd.read_html("Dataset/attacking.html")[0]
defend = pd.read_html("Dataset/defend.html")[0]
pyshic = pd.read_html("Dataset/fisik.html")[0]
goalkeeping = pd.read_html("Dataset/gk.html")[0]
mental = pd.read_html("Dataset/mental.html")[0]
technical = pd.read_html("Dataset/technical.html")[0]
general = pd.read_html("Dataset/general.html")[0]
position = pd.read_html("Dataset/position.html")[0]
contract = pd.read_html("Dataset/contract.html")[0]
#MERGING DATA

print(attack.columns.tolist())
print(goalkeeping.columns.tolist())

source_tables = [attack, defend, pyshic, goalkeeping, mental, technical, general, position, contract]
for table in source_tables:
    table["_name_occurrence"] = table.groupby("Name", dropna=False).cumcount()

merge_keys = ["Name", "_name_occurrence"]
common_columns = ["Rec", "Inf", "WR", "Transfer Value"]
df = general.copy()

for table in [attack, defend, pyshic, goalkeeping, mental, technical, position, contract]:
    table = table.drop(
        columns=[column for column in common_columns if column in table.columns]
    )
    df = df.merge(
        table,
        on=merge_keys,
        how="left",
        validate="one_to_one",
        suffixes=("", "_source")
    )

source_columns = [column for column in df.columns if column.endswith("_source")]
for source_column in source_columns:
    original_column = source_column.removesuffix("_source")
    if original_column in df.columns:
        df[original_column] = df[original_column].combine_first(df[source_column])
        df.drop(columns=source_column, inplace=True)

df = df.drop(columns="_name_occurrence")
df.insert(0, "Player ID", range(1, len(df) + 1))

print(f"Merged rows: {len(df)}")
print(f"Unique player names: {df['Name'].nunique()}")
print(f"Duplicate names from distinct players: {df['Name'].duplicated().sum()}")
print(f"Remaining source columns: {[column for column in df.columns if column.endswith('_source')]}")



df.to_csv("Dataset/fm24_players.csv", index=False)

print(df.head())

['Rec', 'Inf', 'Name', 'Cro', 'Dri', 'Fre', 'Fin', 'Fir', 'Fla', 'Lon', 'OtB', 'Pas', 'Vis', 'Transfer Value']
['Rec', 'Inf', 'Name', 'Aer', 'Cmd', 'Com', 'Ecc', 'Han', 'Kic', '1v1', 'Ref', 'TRO', 'Pun', 'Thr', 'Transfer Value']
Merged rows: 1181
Unique player names: 1175
Duplicate names from distinct players: 6
Remaining source columns: []
   Player ID    Rec  Inf                     Name           Position  \
0          1  - - -  NaN         Brenden Aaronson    M (C), AM (RLC)   
1          2  - - -  NaN           James Abankwah             D (RC)   
2          3  - - -  Wnt                 Ken Aboh             ST (C)   
3          4  - - -  NaN  Kai-Reece Adams-Collman  D (RL), DM, M (C)   
4          5  - - -  Yth              Afi Adebayo    M (R), AM (RLC)   

             Club  Nat  Height Weight  WR  ...  Tec            Best Pos  \
0       Leeds Utd  USA  177 cm  68 kg NaN  ...   13               M (C)   
1         Watford  IRL  183 cm  79 kg NaN  ...   13               D (C)   

In [ ]:
import pandas as pd
import re 

def convert(value):
    if pd.isna(value):
        return 0.0

    matches = re.findall(r"(\d+(?:[.,]\d+)?)\s*([KM])", str(value).upper())
    if not matches:
        return 0.0

    converted_values = []
    for number, unit in matches:
        amount = float(number.replace(",", "."))
        multiplier = 1_000_000 if unit == "M" else 1_000
        converted_values.append(amount * multiplier)

    return max(converted_values)

def wageConvert(value):
    if pd.isna(value):
        return 0.0

    match = re.search(r"[\d.,]+", str(value))
    if not match:
        return 0.0

    number = match.group().replace(",","")
    return float(number)

def changeType(col):
    return col.astype(float)

def drop(df):
    include = ["Rec", "Inf", "WR", "Ability", "Potential"]
    return df.drop(columns=[column for column in include if column in df.columns])



exclude = [
    "Name", "Club", "Nat", "Height", "Weight", "Best Pos", "Best Role",
    "Expires", "Agreed Playing Time", "F/PT", 
]

df = pd.read_csv("Dataset/fm24_players.csv")
df["Transfer Value"] = df["Transfer Value"].apply(convert)
df["Wage"] = df["Wage"].apply(wageConvert)
df = drop(df)



# for col in df.columns:
#     if col in exclude:
#         continue

#     print(col)


# print(df["Pac"].astype(float))

print(df.info())
print(df.iloc[0].to_string())
# print(df["Cro"].dtypes)
# print(df.columns[0])

df.to_csv("Dataset/CleanFm24_players.csv", index=False, encoding="utf-8-sig")

<class 'pandas.DataFrame'>
RangeIndex: 1181 entries, 0 to 1180
Data columns (total 62 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Player ID            1181 non-null   int64  
 1   Name                 1181 non-null   str    
 2   Position             1181 non-null   str    
 3   Club                 1181 non-null   str    
 4   Nat                  1181 non-null   str    
 5   Height               1181 non-null   str    
 6   Weight               1181 non-null   str    
 7   Age                  1181 non-null   int64  
 8   Transfer Value       1181 non-null   float64
 9   Cro                  1181 non-null   int64  
 10  Dri                  1181 non-null   int64  
 11  Fre                  1181 non-null   int64  
 12  Fin                  1181 non-null   int64  
 13  Fir                  1181 non-null   int64  
 14  Fla                  1181 non-null   int64  
 15  Lon                  1181 non-null   int64  
 16 

In [87]:
import pandas as pd

df = pd.read_csv(
    "Dataset/CleanFm24_players.csv",
    encoding="utf-8-sig"
)

target = "Best Role"

exclude = {
    "Player ID",
    "Name",
    "Club",
    "Nat",
    "Position",
    "Best Pos",
    "Best Role",
    "Best Duty",
    "Height",
    "Weight",
    "Expires",
    "Agreed Playing Time",
    "F/PT",
    "Transfer Value"
}

feature_columns = []

for column in df.columns:
    if column in exclude:
        continue

    numeric_column = pd.to_numeric(df[column], errors="coerce")

    if numeric_column.notna().sum() > 0:
        df[column] = numeric_column
        feature_columns.append(column)

X = df[feature_columns].fillna(0)
y = df[target]

valid_rows = y.notna() & y.ne("Unknown")
X = X.loc[valid_rows]
y = y.loc[valid_rows]

print("Jumlah fitur:", len(feature_columns))
print("Daftar fitur:")
print(feature_columns)

print("\nUkuran X:", X.shape)
print("Ukuran y:", y.shape)
print("\nTipe data fitur:")
print(X.dtypes)

Jumlah fitur: 48
Daftar fitur:
['Age', 'Cro', 'Dri', 'Fre', 'Fin', 'Fir', 'Fla', 'Lon', 'OtB', 'Pas', 'Vis', 'Acc', 'Ant', 'Hea', 'Jum', 'Mar', 'Pac', 'Pos', 'Sta', 'Str', 'Tck', 'Agi', 'Bal', 'Aer', 'Cmd', 'Com', 'Ecc', 'Han', 'Kic', '1v1', 'Ref', 'TRO', 'Pun', 'Thr', 'Agg', 'Bra', 'Cmp', 'Cnt', 'Dec', 'Det', 'Ldr', 'Tea', 'Wor', 'Cor', 'L Th', 'Pen', 'Tec', 'Wage']

Ukuran X: (1181, 48)
Ukuran y: (1181,)

Tipe data fitur:
Age       int64
Cro       int64
Dri       int64
Fre       int64
Fin       int64
Fir       int64
Fla       int64
Lon       int64
OtB       int64
Pas       int64
Vis       int64
Acc       int64
Ant       int64
Hea       int64
Jum       int64
Mar       int64
Pac       int64
Pos       int64
Sta       int64
Str       int64
Tck       int64
Agi       int64
Bal       int64
Aer       int64
Cmd       int64
Com       int64
Ecc       int64
Han       int64
Kic       int64
1v1       int64
Ref       int64
TRO       int64
Pun       int64
Thr       int64
Agg       int64
Bra       in

In [95]:
from sklearn.preprocessing import LabelEncoder

label_encoder =LabelEncoder()
y_encoded = label_encoder.fit_transform(y)


In [109]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

role_counts = y.value_counts()
valid_roles = role_counts[role_counts >= 2].index
valid_rows = y.isin(valid_roles)

X_model = X.loc[valid_rows].reset_index(drop=True)
y_model = y.loc[valid_rows].reset_index(drop=True)

label_encoder = LabelEncoder()
y_encoded_model = label_encoder.fit_transform(y_model)

X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y_encoded_model,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded_model
)

model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.03,
    reg_lambda =2,
    subsample=0.8,
    objective="multi:softprob",
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42
)

model.fit(X_train, y_train)
print("Model berhasil dilatih")
print("Jumlah kelas:", len(label_encoder.classes_))
print("Data training:", X_train.shape)
print("Data testing:", X_test.shape)

Model berhasil dilatih
Jumlah kelas: 32
Data training: (942, 48)
Data testing: (236, 48)


In [110]:
from sklearn.metrics import accuracy_score

prediction = model.predict(X_test)
accuracy = accuracy_score(y_test, prediction)

print("Accuracy:", accuracy)

Accuracy: 0.6101694915254238


In [113]:
playstyle_roles = {
    "possession": [
        "Deep Lying Playmaker",
        "Advanced Playmaker",
        "Ball Playing Defender",
        "Inverted Winger"
    ],
    "counter": [
        "Advanced Forward",
        "Winger",
        "Inside Forward",
        "Pressing Forward"
    ],
    "gegenpress": [
        "Pressing Forward",
        "Ball Winning Midfielder",
        "Wing-Back",
        "Central Defender"
    ]
}


def recommend_players(budget, playstyle, top_n=10):
    if playstyle not in playstyle_roles:
        raise ValueError(
            f"Playstyle harus salah satu dari: {list(playstyle_roles)}"
        )

    candidates = df[
        pd.to_numeric(df["Transfer Value"], errors="coerce").fillna(0) <= budget
    ].copy()

    if candidates.empty:
        return pd.DataFrame()

    candidate_features = candidates[feature_columns].fillna(0)
    role_probabilities = model.predict_proba(candidate_features)
    predicted_indexes = role_probabilities.argmax(axis=1)

    candidates["Predicted Role"] = label_encoder.inverse_transform(predicted_indexes)
    candidates["Role Confidence"] = role_probabilities.max(axis=1)

    preferred_roles = [
        role for role in playstyle_roles[playstyle]
        if role in label_encoder.classes_
    ]

    if preferred_roles:
        preferred_indexes = [
            list(label_encoder.classes_).index(role)
            for role in preferred_roles
        ]
        candidates["Playstyle Score"] = role_probabilities[:, preferred_indexes].sum(axis=1)
    else:
        candidates["Playstyle Score"] = candidates["Role Confidence"]

    result_columns = [
        "Name", "Club", "Transfer Value", "Predicted Role",
        "Role Confidence", "Playstyle Score"
    ]
    result_columns = [column for column in result_columns if column in candidates.columns]

    return candidates.sort_values(
        ["Playstyle Score", "Role Confidence"],
        ascending=False
    )[result_columns].head(top_n)


budget = 20_000_000
playstyle = "gegenpress"

recommendations = recommend_players(
    budget=budget,
    playstyle=playstyle,
    top_n=10
)

print(recommendations.to_string(index=False))

                   Name         Club  Transfer Value   Predicted Role  Role Confidence  Playstyle Score
          William Feola         Roma        120000.0 Central Defender         0.997588         0.997794
            Zach Simons    Tottenham         90000.0 Central Defender         0.996894         0.997229
            Josh Briggs     West Ham        120000.0 Central Defender         0.996681         0.997024
            Noah McCann          QPR         40000.0 Central Defender         0.996050         0.996406
              Luke Bell   Sunderland        140000.0 Central Defender         0.995919         0.996277
           David Modupe Nottm Forest        350000.0 Central Defender         0.995426         0.995999
Pele Arganese-McDermott    Tottenham        160000.0 Central Defender         0.995066         0.995481
             Alfie Pond       Wolves        180000.0 Central Defender         0.995035         0.995312
           Matas Klimas    Brentford        140000.0 Central Def